# Daily Citi Bike demand: feature importance and SHAP

Explain a **frozen monthly model from MLflow** predicting daily trip starts per station.
This companion follows the setup → validate → chronological split → model diagnostics →
charts → interpretation structure of `linear_regression_demand_forecasting.ipynb`.
It does **not** retrain, tune, promote, or register models.

| Data | Role |
|---|---|
| Saved model's training window | Already used to fit the model; never refitted here |
| Last 28 training days | Sample a SHAP reference background, without labels |
| Month starting on the configured snapshot date | Held-out evaluation and explanation |

**MAE is measured in trips per station per day.** Permutation importance measures a
change in MAE; SHAP attributes predicted trip counts. Those are different questions,
so their magnitudes must not be compared directly. The winner can be Poisson regression,
random forest, or XGBoost; the notebook is not hard-coded to XGBoost.

## Setup and configuration

Run the dependency cell below once in this notebook's selected kernel, then choose
**Restart Kernel** and run the remaining cells. `%pip` targets the active notebook
kernel, which may differ from your terminal's Python or the Docker image.
The SHAP version matches `requirements.txt`.

For a new environment, install the complete `requirements-notebooks.txt` instead.
Do not install project packages into Airflow's base environment.

The defaults explain the **historical champion effective on September 1, 2017**, evaluated
on September 1–30, 2017. Change `SNAPSHOT_DATE` to explain another month's champion.
Selection requires a published, promoted model eligible on that date; a rejected candidate
is never used as a fallback. The chosen model stays frozen across the evaluation window.
Override `EVALUATION_END` for a shorter period. Evaluation must follow the model's cutoff.

| Calculation | Default sample | Project alignment |
|---|---:|---|
| Evaluation MAE/RMSE/R² | All station-days in the evaluation window | Full-window scoring |
| Individual and grouped permutation importance | Up to 1,000 station-days, 3 shuffles | Training importance budget |
| SHAP explanations | Up to 2,000 station-days | Monthly SHAP explanation budget |
| SHAP reference background | Up to 32 earlier training station-days | Notebook-specific reference |

Production predictions are made for all eligible stations; 1,000 and 2,000 are explanation
sample limits, not prediction limits. Matching the sample budgets does not make the
methods or evaluation windows identical to the scheduled jobs.


In [ ]:
# Run once if SHAP is missing, then restart the kernel.
%pip install shap==0.49.1


In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import shap
except ImportError as error:
    raise ImportError("Install requirements-notebooks.txt into this kernel, then restart it.") from error

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "utils/model_store.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Launch this notebook from the repository root or notebooks/.")
sys.path.insert(0, str(ROOT))

import mlflow
from utils.model_store import ModelStore
from utils.feature_importance import measure_feature_importance

SNAPSHOT_DATE = "2017-09-01"  # Select the historical champion effective on this date.
EVALUATION_END = None   # Inclusive; default is the snapshot month's last day.
RANDOM_STATE = 42
PERMUTATION_ROWS = 1000
N_REPEATS = 3
SHAP_ROWS = 2000       # Match the monthly explanation job's row budget.
BACKGROUND_ROWS = 32
SHAP_PERMUTATIONS = 3
MAX_DISPLAY = 15
SHAP_MAX_DISPLAY = 9  # Eight named features plus an aggregated other-features row.
SHAP_FONT_SIZE = 16
EXPORT = False

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME", "citibike-demand-forecasting")
GOLD = ROOT / "datamart/gold"
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
figures = {}

def shap_slide_footer():
    return (
        f"Model version: {saved.version} | Evaluation: {evaluation_start:%d %b %Y}–{evaluation_end:%d %b %Y}"
        f" | SHAP sample: {len(X_explain):,} station-days"
    )


def show_shap_chart(name, figure):
    from matplotlib.text import Text
    # SHAP assigns font sizes internally, so resize its labels after plotting.
    for text in figure.findobj(match=Text):
        text.set_fontsize(SHAP_FONT_SIZE)
    # Waterfall uses tightly positioned twin-axis labels for the reference/output.
    for axis in figure.axes[1:]:
        for text in axis.get_xticklabels():
            text.set_fontsize(12)
    figure.text(0.01, -0.035, shap_slide_footer().replace(" | Evaluation:", "\nEvaluation:"),
                fontsize=12, color="#526174", va="top")
    figures[name] = figure
    plt.show()
    plt.close(figure)


def show_chart(name, figure):
    figures[name] = figure
    plt.show()

## Select a completed MLflow model

List completed, published monthly winners first. The normal selection uses the
**historical champion effective on `SNAPSHOT_DATE`**, not today's registry alias or
newest model version. If no promoted model is eligible, selection fails explicitly.

Keep the run ID and model URI with exported results so charts remain attributable
when future rolling updates choose another algorithm. The existing experiment must
exist; this notebook does not create training runs or models.

In [ ]:
mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient(tracking_uri=TRACKING_URI)
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None or experiment.lifecycle_stage != "active":
    raise ValueError("No active Citi Bike experiment found. Run monthly training first.")
store = ModelStore(tracking_uri=TRACKING_URI, experiment_name=EXPERIMENT_NAME)
available = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED' AND tags.`citibike.ready` = 'true'",
    max_results=10000,
)
if available.empty:
    raise ValueError("No completed, published models found. Run monthly training first.")
display(available[["run_id", "tags.model_type", "tags.trained_through", "tags.effective_from",
                   "tags.promoted", "tags.model_uri"]].sort_values("tags.trained_through"))

saved = store.champion(SNAPSHOT_DATE, inclusive=True)
if saved is None:
    raise ValueError(f"No published, promoted champion effective on {SNAPSHOT_DATE}")
model = store.load(saved)
info = store.info(saved)
feature_columns = info["features"]
label = info["label"]
cutoff = pd.Timestamp(saved.trained_through)
evaluation_start = pd.Timestamp(SNAPSHOT_DATE)
evaluation_end = pd.Timestamp(EVALUATION_END) if EVALUATION_END else evaluation_start + pd.offsets.MonthEnd(0)
if evaluation_start <= cutoff or evaluation_end < evaluation_start:
    raise ValueError("Evaluation must be strictly after training and have a non-empty date range.")
if evaluation_start < pd.Timestamp(saved.effective_from):
    raise ValueError("This model was not yet effective on the requested evaluation date.")
background_start = max(pd.Timestamp(info["trained_from"]), cutoff - pd.Timedelta(days=27))
print(f"Historical champion on {SNAPSHOT_DATE}: {saved.version} | algorithm: {info['model']} | promoted: {saved.promoted}")
print(f"Effective from: {saved.effective_from} | evaluation: {evaluation_start.date()} to {evaluation_end.date()}")
print(f"Run: {saved.run_id} | URI: {saved.model_uri}")

## Load and validate the daily gold stores

Unlike the reference notebook's separate monthly combined stores, production models
use **daily `datamart/gold` partitions**. Read them with pandas, without starting Spark.
Reject missing partitions, duplicate keys, unmatched feature/label rows, invalid targets,
and missing model inputs. The saved feature list guarantees the same input order.

Date separation prevents fitting on the explanation month. It does not independently
prove input availability: this analysis inherits the production pipeline's weather and
trip-feature publication assumptions. Verify those before calling a result operationally
leakage-free. No gold files or model artifacts are modified.

In [ ]:
KEYS = ["station_id", "snapshot_date"]

def read_daily_store(directory, prefix, start, end, columns):
    frames = []
    for day in pd.date_range(start, end, freq="D"):
        path = GOLD / directory / f"{prefix}_{day:%Y_%m_%d}.parquet"
        if not path.is_dir():
            raise FileNotFoundError(f"Missing {path}; build the gold stores or shorten the date range.")
        frame = pd.read_parquet(path, columns=list(dict.fromkeys(KEYS + columns)))
        frame["snapshot_date"] = pd.to_datetime(frame["snapshot_date"])
        if frame.empty or not frame["snapshot_date"].eq(day).all():
            raise ValueError(f"Empty or incorrectly dated partition: {path}")
        frames.append(frame)
    result = pd.concat(frames, ignore_index=True)
    if result[KEYS].isna().any().any() or result.duplicated(KEYS).any():
        raise ValueError(f"Invalid or duplicate station/date keys in {directory}")
    return result.sort_values(["snapshot_date", "station_id"]).reset_index(drop=True)

background_data = read_daily_store("feature_store", "gold_feature_store", background_start, cutoff, feature_columns)
evaluation_features = read_daily_store("feature_store", "gold_feature_store", evaluation_start, evaluation_end, feature_columns)
evaluation_labels = read_daily_store("label_store", "gold_label_store", evaluation_start, evaluation_end, [label, "label_def"])
if set(evaluation_labels["label_def"].dropna().unique()) != {"trip_starts_1d"}:
    raise ValueError("Expected daily trip-start labels")
evaluation = evaluation_features.merge(evaluation_labels, on=KEYS, how="outer", validate="one_to_one", indicator=True)
if not evaluation["_merge"].eq("both").all():
    raise ValueError(f"Unmatched keys: {evaluation['_merge'].value_counts().to_dict()}")
evaluation = evaluation.drop(columns="_merge").sort_values(["snapshot_date", "station_id"]).reset_index(drop=True)
for name, frame in [("background", background_data), ("evaluation", evaluation)]:
    if frame[feature_columns].isna().any().any():
        raise ValueError(f"Null model inputs in {name}")
    numeric = frame[feature_columns].select_dtypes(include="number")
    if not np.isfinite(numeric.to_numpy()).all():
        raise ValueError(f"Non-finite model inputs in {name}")
if evaluation[label].isna().any() or not np.isfinite(evaluation[label]).all() or evaluation[label].lt(0).any():
    raise ValueError("Targets must be finite, non-null, and non-negative")
assert background_data.snapshot_date.max() < evaluation.snapshot_date.min()
display(evaluation.head())

## Separate the reference and explanation samples chronologically

Background observations come from the end of the **training period**; they define the
SHAP reference distribution. Explanation observations come only from the later evaluation
period. Sampling is reproducible, positional labels stay aligned, and each method has a
small explicit budget. No fitting or randomized model search is performed.

In [ ]:
summary = pd.DataFrame([
    {"role": "SHAP background candidates", "rows": len(background_data),
     "first_date": background_data.snapshot_date.min(), "last_date": background_data.snapshot_date.max()},
    {"role": "Held-out evaluation", "rows": len(evaluation),
     "first_date": evaluation.snapshot_date.min(), "last_date": evaluation.snapshot_date.max()},
]).set_index("role")
display(summary)
background = background_data.sample(n=min(BACKGROUND_ROWS, len(background_data)), random_state=RANDOM_STATE)
importance_rows = np.random.default_rng(RANDOM_STATE).choice(len(evaluation), min(PERMUTATION_ROWS, len(evaluation)), replace=False)
importance_data = evaluation.iloc[importance_rows]
explain_data = evaluation.sample(n=min(SHAP_ROWS, len(evaluation)), random_state=RANDOM_STATE)
X_background = background[feature_columns]
X_importance, y_importance = importance_data[feature_columns], importance_data[label]
X_explain = explain_data[feature_columns]
print(f"Permutation rows: {len(X_importance):,}; SHAP rows: {len(X_explain)}; background rows: {len(X_background)}")

## Evaluate the frozen model before interpreting it

Feature importance does not prove that a model predicts well. Compare the selected
pipeline with the seven-day-lag baseline on the full held-out window, then inspect
performance over time. These diagnostics are retrospective; do not tune parameters
against this month after treating it as an untouched evaluation.

In [ ]:
evaluation_predictions = evaluation[KEYS + [label]].copy()
evaluation_predictions["Model"] = model.predict(evaluation[feature_columns])
evaluation_predictions["Seven-day lag"] = evaluation["fe_lag_7"].to_numpy()
metrics = pd.DataFrame([
    {"model": name, "MAE": mean_absolute_error(evaluation[label], evaluation_predictions[name]),
     "RMSE": np.sqrt(mean_squared_error(evaluation[label], evaluation_predictions[name])),
     "R2": r2_score(evaluation[label], evaluation_predictions[name])}
    for name in ["Model", "Seven-day lag"]
]).set_index("model")
display(metrics)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
metrics["MAE"].plot.bar(ax=axes[0], rot=0, color=["steelblue", "grey"])
axes[0].set(title="Held-out MAE", ylabel="Trips per station/day", xlabel="")
daily = evaluation_predictions.groupby("snapshot_date")[[label, "Model", "Seven-day lag"]].sum()
daily.rename(columns={label: "Actual"}).plot(ax=axes[1], linewidth=1)
axes[1].set(title="Demand across active stations", ylabel="Daily trip starts", xlabel="Date")
show_chart("evaluation", fig)

## Individual permutation importance: which original inputs help accuracy?

Shuffle each original input in the **whole frozen pipeline** and measure the increase
in MAE. This includes `station_id` as one input affecting both station-profile features.
Higher positive values indicate greater reliance; retain negative values rather than
clamping them. Error bars are standard deviations across shuffles, **not confidence
intervals**. Correlated trip lags and weather variables can substitute for one another.

This reuses the scheduled training helper but evaluates a different model/window: the
**saved final model on a later month**. The importance already logged during training
instead described the pre-final-refit model on its earlier test month. Do not conflate them.

In [ ]:
permutation_report = measure_feature_importance(
    model, evaluation[feature_columns], evaluation[label], seed=RANDOM_STATE,
    max_rows=PERMUTATION_ROWS, n_repeats=N_REPEATS,
)
feature_importance = pd.DataFrame(permutation_report["features"])
print(f"Sample baseline MAE: {permutation_report['sample_baseline_mae']:.4f}")
display(feature_importance)
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
top = feature_importance.head(MAX_DISPLAY).sort_values("importance_mean")
ax.barh(top.feature, top.importance_mean, xerr=top.importance_std, capsize=3)
ax.axvline(0, color="grey", linewidth=1)
ax.set(title="Individual permutation importance", xlabel="Increase in held-out MAE after shuffling")
show_chart("permutation_individual", fig)

## Grouped permutation importance: correlated inputs together

Shuffle each feature family using **one shared row permutation per repeat**. That preserves
relationships within the family while breaking its alignment with the prediction target.
The groups cover every original input exactly once. This is a diagnostic association,
not a causal intervention; combinations across groups can still become unrealistic.

In [ ]:
feature_groups = {
    "Trip history": [c for c in feature_columns if c.startswith(("fe_lag_", "fe_rolling_"))],
    "Calendar": [c for c in feature_columns if c.startswith("fe_") and not c.startswith(("fe_lag_", "fe_rolling_"))],
    "Weather": [c for c in feature_columns if not c.startswith("fe_") and c != "station_id"],
    "Station identity": ["station_id"],
}
grouped_columns = [c for columns in feature_groups.values() for c in columns]
if set(grouped_columns) != set(feature_columns) or len(grouped_columns) != len(set(grouped_columns)):
    raise ValueError("Feature groups must cover every input exactly once")
group_baseline_mae = mean_absolute_error(y_importance, model.predict(X_importance))
rng = np.random.default_rng(RANDOM_STATE)
group_records = []
for group, columns in feature_groups.items():
    increases = []
    for _ in range(N_REPEATS):
        shuffled = X_importance.copy()
        order = rng.permutation(len(shuffled))
        for column in columns:
            shuffled[column] = X_importance[column].to_numpy()[order]
        increases.append(mean_absolute_error(y_importance, model.predict(shuffled)) - group_baseline_mae)
    group_records.append({"group": group, "MAE increase": np.mean(increases), "shuffle std": np.std(increases)})
group_importance = pd.DataFrame(group_records).sort_values("MAE increase", ascending=False)
display(group_importance)
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
ordered = group_importance.sort_values("MAE increase")
ax.barh(ordered["group"], ordered["MAE increase"], xerr=ordered["shuffle std"], capsize=3)
ax.axvline(0, color="grey", linewidth=1)
ax.set(title="Grouped permutation importance", xlabel="Increase in held-out MAE after shuffling")
show_chart("permutation_grouped", fig)

## Transform inputs without refitting

SHAP and native estimator diagnostics below operate **after frozen preprocessing**.
For tree models, `station_id` becomes `station_mean` and `station_weekend_ratio`.
For Poisson regression, the additional fitted column transformer supplies weekday
one-hot columns, logged/scaled counts, and standardized remaining inputs.

Keep transformed names in exactly the fitted output order. The counts transform is
one-to-one even though its `FunctionTransformer` does not declare feature names.
Only the two current production pipeline layouts are supported; fail visibly for an
unexpected layout. Small sampled matrices may be densified, never the full gold data.

In [ ]:
def transformed_inputs(pipeline, frame):
    transformed = pipeline[:-1].transform(frame)
    if isinstance(transformed, pd.DataFrame):
        return transformed.astype(float)
    if len(pipeline.steps) != 3:
        raise ValueError("Expected StationProfile + optional ColumnTransformer + regressor")
    profiled = pipeline.steps[0][1].transform(frame)
    column_transformer = pipeline.steps[1][1]
    names = []
    for group, transformer, columns in column_transformer.transformers_:
        if isinstance(transformer, str) and transformer == "drop":
            continue
        columns = list(columns)
        if not columns:
            continue
        if isinstance(columns[0], (int, np.integer)):
            columns = [profiled.columns[i] for i in columns]
        if isinstance(transformer, str) and transformer == "passthrough":
            outputs = columns
        else:
            try:
                outputs = list(transformer.get_feature_names_out(columns))
            except AttributeError:
                # The production log1p + StandardScaler counts pipeline is one-to-one.
                width = transformer.transform(profiled[columns]).shape[1]
                if width != len(columns):
                    raise ValueError(f"Cannot name transformed columns for {group}")
                outputs = columns
        names.extend(f"{group}__{name}" for name in outputs)
    values = transformed.toarray() if sparse.issparse(transformed) else np.asarray(transformed)
    if values.shape[1] != len(names):
        raise ValueError("Transformed names do not match estimator input width")
    return pd.DataFrame(values, columns=names, index=frame.index).astype(float)

estimator = model.steps[-1][1]
background_transformed = transformed_inputs(model, X_background)
explain_transformed = transformed_inputs(model, X_explain)
transformed_names = explain_transformed.columns.tolist()
if background_transformed.columns.tolist() != transformed_names:
    raise ValueError("Background and explanation transformations disagree")

def predict_transformed(values):
    frame = pd.DataFrame(values, columns=transformed_names)
    inputs = frame if hasattr(estimator, "feature_names_in_") else frame.to_numpy()
    return estimator.predict(inputs)

np.testing.assert_allclose(predict_transformed(explain_transformed.to_numpy()), model.predict(X_explain), rtol=1e-6, atol=1e-6)
display(explain_transformed.head())
print(f"SHAP uses {len(transformed_names)} transformed inputs; permutation uses {len(feature_columns)} original inputs.")

## Native feature importance or coefficients

For **XGBoost**, use its configured split importance (gain by default). For **random
forest**, use normalized impurity reduction. Both are training-based diagnostics, can
favor particular feature structures, and are not held-out accuracy improvements.
For **Poisson regression**, show signed coefficients on transformed inputs: they act on
log expected demand, and differing transformed scales limit magnitude comparisons.
These values are not interchangeable with permutation MAE changes or SHAP trip counts.

In [ ]:
if hasattr(estimator, "feature_importances_"):
    native_values = np.asarray(estimator.feature_importances_)
    if hasattr(estimator, "get_booster"):
        importance_type = estimator.get_params().get("importance_type") or "gain"
        native_title = f"XGBoost native importance ({importance_type})"
    else:
        native_title = "Random forest native importance (impurity reduction)"
    native_units = "Normalized estimator importance"
elif hasattr(estimator, "coef_"):
    native_values = np.asarray(estimator.coef_).ravel()
    native_title = "Poisson regression coefficients on transformed inputs"
    native_units = "Signed coefficient (log expected trips; input scales differ)"
else:
    raise TypeError(f"No native importance available for {type(estimator).__name__}")
native_importance = pd.DataFrame({"feature": transformed_names, "value": native_values})
native_importance["absolute value"] = native_importance.value.abs()
native_importance = native_importance.sort_values("absolute value", ascending=False)
display(native_importance)
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
ordered = native_importance.head(MAX_DISPLAY).sort_values("value")
ax.barh(ordered.feature, ordered.value)
ax.axvline(0, color="grey", linewidth=1)
ax.set(title=native_title, xlabel=native_units)
show_chart("native_importance", fig)

## SHAP: explain predictions in trip-count units

Use **model-agnostic permutation SHAP** for every winning algorithm. Explaining the final
estimator's `predict` function avoids XGBoost Poisson's raw-log-margin versus trip-count
ambiguity. The background comes only from earlier training dates. Frozen preprocessing
has already been applied; in this explanation space the two learned station-profile
features are separate inputs, not raw `station_id`.

The default budget is up to 2,000 rows, 32 background observations, and three permutation
cycles per row. This explains about 31 times as many rows as the old 64-row preview,
so expect a longer run. More rows improve population coverage; the background and
permutation-cycle budgets still affect approximation stability.

The scheduled `model_explain.py` also samples 2,000 rows, but uses Tree/Linear SHAP on
each final refitted candidate's former test month. This notebook explains the historical
champion on a later month using permutation SHAP in trip-count units; scores are not
interchangeable.

SHAP sums to predicted trips relative
to this specific background, **not** to prediction error or feature causality. Masking
can create implausible combinations of correlated inputs. Seeded results are reproducible
but are still approximate; rankings should not drive retuning on the held-out month.

In [ ]:
explainer = shap.Explainer(
    predict_transformed, background_transformed.to_numpy(),
    feature_names=transformed_names, algorithm="permutation", seed=RANDOM_STATE,
)
shap_values = explainer(
    explain_transformed.to_numpy(),
    max_evals=SHAP_PERMUTATIONS * (2 * len(transformed_names) + 1),
    silent=False,
)
reconstructed = np.asarray(shap_values.base_values).reshape(-1) + shap_values.values.sum(axis=1)
np.testing.assert_allclose(reconstructed, model.predict(X_explain), rtol=1e-5, atol=1e-5)
shap_importance = pd.DataFrame({
    "feature": transformed_names,
    "mean absolute SHAP (trips)": np.abs(shap_values.values).mean(axis=0),
}).sort_values("mean absolute SHAP (trips)", ascending=False)
display(shap_importance)
print("SHAP additivity verified against full-pipeline predictions in trip-count units.")

## Global SHAP charts

The bar chart ranks average absolute contributions across the sampled station/day rows.
The beeswarm adds direction: positive SHAP raises predicted demand above the background
expectation, negative SHAP lowers it. Colors show transformed feature values, not causal
strength. Station-profile values are learned historical summaries, not raw station IDs.
Presentation exports show eight named features plus a combined “other features” row.
Labels use 16-point text. The footer is generated from the selected model, evaluation
window, and actual explanation sample size; it is also saved as `slide_footer.txt`.


In [ ]:
with plt.rc_context({"font.size": SHAP_FONT_SIZE}):
    shap.plots.bar(shap_values, max_display=SHAP_MAX_DISPLAY, show=False)
    plt.gcf().set_size_inches(12, 8)
    show_shap_chart("shap_global_bar", plt.gcf())
    shap.plots.beeswarm(shap_values, max_display=SHAP_MAX_DISPLAY,
                        plot_size=(12, 8), show=False)
    show_shap_chart("shap_beeswarm", plt.gcf())


## Local SHAP waterfall: explain one station/day

Choose a positional row from the reproducible explanation sample. The waterfall starts
at the background expected prediction and adds feature contributions to reach this
row's predicted daily trips. The actual count is shown for context, but SHAP does not
explain the residual. No special high-error or high-demand row is selected automatically.

In [ ]:
ROW_POSITION = 0
if not 0 <= ROW_POSITION < len(explain_data):
    raise IndexError("ROW_POSITION must be inside the explanation sample")
row = explain_data.iloc[ROW_POSITION]
print(f"Station {row.station_id} | date {row.snapshot_date.date()} | actual {row[label]:.0f}")
print(f"Prediction: {model.predict(X_explain.iloc[[ROW_POSITION]])[0]:.2f} trips")
with plt.rc_context({"font.size": SHAP_FONT_SIZE}):
    shap.plots.waterfall(shap_values[ROW_POSITION], max_display=SHAP_MAX_DISPLAY, show=False)
    plt.gcf().set_size_inches(12, 8.5)
    show_shap_chart("shap_waterfall", plt.gcf())
print("Slide footer:", shap_slide_footer())

## SHAP dependence: how does a feature relate to its contribution?

Inspect the highest-ranked transformed feature, or override `DEPENDENCE_FEATURE`.
Each dot is a sampled station/day. Other inputs and interactions affect its contribution,
so this is not a controlled what-if experiment, a causal response, or a confidence band.

In [ ]:
DEPENDENCE_FEATURE = shap_importance.iloc[0]["feature"]
if DEPENDENCE_FEATURE not in transformed_names:
    raise ValueError("Choose a transformed feature shown in shap_importance")
shap.plots.scatter(shap_values[:, DEPENDENCE_FEATURE], show=False)
show_chart("shap_dependence", plt.gcf())

## Read the results and export charts

- **Permutation MAE:** which original inputs help held-out accuracy?
- **Grouped permutation:** how much does a correlated family help collectively?
- **Native tree importance / coefficients:** what structure did the fitted estimator use?
- **SHAP:** which transformed inputs contribute to predicted trip counts globally and locally?
- Compare the story, not the numeric scales or raw ranks across different feature spaces.
  A strong station-ID permutation score can coexist with two separate station-profile SHAP scores.
- Check dependence on sample size, seed, background period, and season before drawing conclusions.
  Do not treat correlated features or SHAP contributions as causal effects.
- Rolling updates can change both the winner and the champion. Repeat this notebook with another
  configured snapshot date; keep the run ID and model URI attached to every result.

Set `EXPORT = True` in setup and rerun the cell below to save figures, tables, sampled
SHAP values, and provenance locally. Nothing is uploaded to MLflow; no existing training
run, registry version, or gold partition is changed. The notebook keeps useful variables
(`metrics`, `feature_importance`, `group_importance`, `native_importance`, `shap_importance`,
`shap_values`, and `evaluation_predictions`) available for further analysis.

In [ ]:
if EXPORT:
    output = ROOT / "reports/feature_importance" / saved.run_id / f"{evaluation_start:%Y-%m-%d}_{evaluation_end:%Y-%m-%d}"
    output.mkdir(parents=True, exist_ok=True)
    for name, figure in figures.items():
        figure.savefig(output / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    (output / "slide_footer.txt").write_text(shap_slide_footer() + "\n", encoding="utf-8")
    for name, table in {
        "metrics": metrics.reset_index(), "permutation": feature_importance,
        "grouped_permutation": group_importance, "native": native_importance, "shap": shap_importance,
        "shap_sample": explain_data[KEYS + [label]],
    }.items():
        table.to_csv(output / f"{name}.csv", index=False)
    np.savez_compressed(output / "shap_values.npz", values=shap_values.values,
                        base_values=shap_values.base_values, data=shap_values.data,
                        feature_names=np.asarray(transformed_names, dtype=str))
    provenance = {
        "run_id": saved.run_id, "model_version": saved.version, "model_uri": saved.model_uri, "model_type": info["model"],
        "model_selection": "historical_champion", "selection_date": SNAPSHOT_DATE,
        "effective_from": saved.effective_from,
        "trained_through": saved.trained_through, "evaluation_start": str(evaluation_start.date()),
        "evaluation_end": str(evaluation_end.date()), "background_start": str(background_start.date()),
        "background_end": str(cutoff.date()), "seed": RANDOM_STATE, "shap_version": shap.__version__,
        "permutation_rows": len(X_importance), "permutation_repeats": N_REPEATS,
        "shap_rows": len(X_explain), "background_rows": len(X_background),
        "shap_permutations": SHAP_PERMUTATIONS, "shap_units": "predicted trips per station/day",
        "shap_input_space": "frozen transformed estimator inputs",
        "shap_max_display": SHAP_MAX_DISPLAY, "shap_font_size": SHAP_FONT_SIZE,
        "waterfall_station_id": str(row.station_id), "waterfall_date": str(row.snapshot_date.date()),
        "waterfall_reference": float(np.asarray(shap_values.base_values).reshape(-1)[ROW_POSITION]),
        "waterfall_prediction": float(reconstructed[ROW_POSITION]),
        "slide_footer": shap_slide_footer(),
    }
    (output / "provenance.json").write_text(json.dumps(provenance, indent=2))
    print(f"Saved {len(figures)} charts and analysis tables to {output}")
else:
    print("EXPORT=False: charts are displayed inline only; MLflow and gold data are unchanged.")